# Analisis Kerawanan Banjir Kota Palu (Level 2: AHP Hydrologic Weighting)
### Komparasi K-Means dan Fuzzy C-Means Berbasis SIG dengan Gaussian Quantile & Bobot Hidrologi
**Peneliti:** Melsy Patricia Anggelina. E (F551 21 026)  
**Program Studi:** S1 Teknik Informatika, Universitas Tadulako

Notebook ini memuat pipeline eksperimen teroptimasi (**Level 2: AHP Hydrologic**) untuk analisis kerawanan banjir di Kota Palu:
1. Pembersihan data (*data cleaning*) dan penanganan nilai kosong.
2. Transformasi numerik penggunaan lahan (*land use runoff scoring*).
3. Optimasi preprocessing: *Quantile Gaussian Transformation*, *Min-Max Scaling*, dan pembobotan parameter (Elevasi 30%, Penggunaan Lahan 30%, Kemiringan Lereng 20%, Curah Hujan 10%, Jarak Sungai 10%).
4. Penentuan jumlah klaster optimal dengan *Elbow Method* dan *Silhouette Score* ($k \in [2, 6]$).
5. Pelatihan model komparatif: K-Means vs Fuzzy C-Means.
6. Evaluasi komparatif lintas $k$ dan ringkasan akurasi model dalam bentuk persentase ($\%$).
7. Profiling karakteristik fisik dan penentuan tingkat kerawanan (Tinggi, Sedang, Rendah).
8. Analisis keunggulan *soft clustering* FCM melalui deteksi zona transisi (peringatan dini).
9. Visualisasi spasial sebaran kerawanan banjir Kota Palu (UTM Zona 50S).
10. Ekspor hasil CSV dan penyimpanan model mandiri terintegrasi (2 file `.pkl`).
11. Pengujian inferensi langsung pada data baru.


## 1. Instalasi Dependensi & Import Pustaka

In [ ]:
# scikit-fuzzy requires packaging to resolve version checks
!pip install -q scikit-fuzzy packaging joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.metrics import davies_bouldin_score, silhouette_score
import skfuzzy as fuzz

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

## 2. Pemuatan Dataset Spasial

In [ ]:
possible_paths = [
    "dataset_banjir_palu_final3.csv",
    "resources/dataset_banjir_palu_final3.csv",
    "../resources/dataset_banjir_palu_final3.csv",
    "/content/dataset_banjir_palu_final3.csv"
]

dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    try:
        from google.colab import files
        print("Dataset tidak ditemukan di path lokal. Silakan upload file dataset_banjir_palu_final3.csv:")
        uploaded = files.upload()
        dataset_path = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("File dataset_banjir_palu_final3.csv tidak ditemukan.")

df_raw = pd.read_csv(dataset_path)
print(f"Dataset berhasil dimuat: {df_raw.shape[0]} baris sampel, {df_raw.shape[1]} kolom.")
df_raw.head()

## 3. Pembersihan Data (Data Cleaning)
Memeriksa *missing values*, string kosong `""`, dan memastikan tipe data numerik pada 5 parameter fisik.

In [ ]:
df_clean = df_raw.copy()
df_clean = df_clean.replace(r"^\s*$", np.nan, regex=True)

missing_report = df_clean.isnull().sum()
print("Jumlah Missing Values per Kolom:")
print(missing_report[missing_report > 0])

# Drop records missing critical flood attributes (rows 905, 1426, 1427)
df_clean = df_clean.dropna(subset=['elev_fix_1', 'slope_fix_1', 'REMARK']).copy()

df_clean['elev_fix_1'] = pd.to_numeric(df_clean['elev_fix_1'])
df_clean['slope_fix_1'] = pd.to_numeric(df_clean['slope_fix_1'])
df_clean['hujan_fix_1'] = pd.to_numeric(df_clean['hujan_fix_1'])
df_clean['distance'] = pd.to_numeric(df_clean['distance'])

print(f"\nJumlah sampel bersih setelah pembersihan: {len(df_clean)} titik sampel.")

## 4. Transformasi Variabel Penggunaan Lahan (Land Use Scoring)
Data kategorikal penggunaan lahan (`REMARK`) dikonversi ke skor numerik (1 - 5) berdasarkan kapasitas infiltrasi dan limpasan permukaan:
- **Skor 5 (Sangat Rawan / Kedap Air):** Permukiman, Industri, Pelabuhan.
- **Skor 4 (Rawan / Badan Air & Terbuka):** Tanah Kosong/Gundul, Sungai.
- **Skor 3 (Sedang):** Sawah, Tegalan/Ladang.
- **Skor 2 (Rendah):** Perkebunan, Semak Belukar, Pasir/Bukit Pasir Laut, Tanaman Campur.
- **Skor 1 (Sangat Rendah / Resapan Maksimal):** Hutan Rimba, Pemakaman Umum.

In [ ]:
# Hydrological runoff weight: 1 (highest infiltration) to 5 (impervious surface)
landuse_scoring = {
    'Hutan Rimba': 1,
    'Pemakaman Umum': 1,
    'Semak Belukar': 2,
    'Perkebunan/Kebun': 2,
    'Tanaman Campur': 2,
    'Pasir/Bukit Pasir Laut': 2,
    'Tegalan/Ladang': 3,
    'Sawah': 3,
    'Tanah Kosong/Gundul': 4,
    'Sungai': 4,
    'Permukiman dan Tempat Kegiatan': 5,
    'Industri Manufaktur Lainnya': 5,
    'Pelabuhan': 5
}

df_clean['landuse_score'] = df_clean['REMARK'].map(landuse_scoring)

print("Distribusi Skor Penggunaan Lahan:")
print(df_clean['landuse_score'].value_counts().sort_index())

## 5. Optimasi Preprocessing & Pembobotan (Level 2: AHP Hydrologic)
Tahap ini mengatasi *skewness* ekstrem pada data jarak sungai dan kontras lembah Palu melalui 3 langkah:
1. **Quantile Gaussian Transformation:** Memetakan distribusi data menceng ke kurva normal agar ruang ukur Euclidean tidak bias.
2. **Min-Max Scaling:** Membatasi skala fitur ke rentang $[0, 1]$.
3. **Pembobotan Parameter Fisik:** Elevasi 30%, Penggunaan Lahan 30%, Kemiringan Lereng 20%, Curah Hujan 10%, Jarak Sungai 10%.

Kolom spasial (`X`, `Y`) dan identitas (`id`, `Shape_Length`, `Shape_Area`) dikecualikan dari matriks klasterisasi.

In [ ]:
physical_features = ['hujan_fix_1', 'elev_fix_1', 'slope_fix_1', 'distance', 'landuse_score']
X_raw = df_clean[physical_features].values

# 1. Quantile Gaussian mapping
qt_transformer = QuantileTransformer(output_distribution='normal', random_state=42)
X_gaussian = qt_transformer.fit_transform(X_raw)

# 2. Min-Max normalization to [0, 1]
scaler_opt = MinMaxScaler()
X_norm = scaler_opt.fit_transform(X_gaussian)

# 3. Apply feature weights
weights = np.array([0.10, 0.30, 0.20, 0.10, 0.30])
X_weighted = X_norm * weights

df_transformed = pd.DataFrame(X_weighted, columns=[f"{col}_opt" for col in physical_features])
print("Ringkasan Statistik Fitur Setelah Optimasi & Pembobotan:")
display(df_transformed.describe().T[['mean', 'std', 'min', 'max']])

## 6. Penentuan Jumlah Klaster Optimal (Elbow Method & Silhouette Score)
Pengujian rentang $k \in [2, 6]$ dilakukan untuk menentukan jumlah zona paling terpisah dan stabil.

In [ ]:
k_range = range(2, 7)
inertias = []
silhouette_scores = []
dbi_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_weighted)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_weighted, labels))
    dbi_scores.append(davies_bouldin_score(X_weighted, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, marker='o', color='#1976D2', linewidth=2)
axes[0].set_title('Elbow Method (Inersia / SSE)')
axes[0].set_xlabel('Jumlah Klaster (k)')
axes[0].set_ylabel('Within-Cluster Sum of Squares (Inersia)')
axes[0].set_xticks(list(k_range))

axes[1].plot(k_range, silhouette_scores, marker='s', color='#388E3C', linewidth=2)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Jumlah Klaster (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.show()

# Use k=3 matching proposal vulnerability tiers (Rendah, Sedang, Tinggi)
optimal_k = 3
print(f"Jumlah klaster yang digunakan sesuai zonasi risiko proposal: k = {optimal_k}")
print(f"  Silhouette Score pada k={optimal_k}: {silhouette_scores[optimal_k - 2]:.4f}")
print(f"  Davies-Bouldin Index pada k={optimal_k}: {dbi_scores[optimal_k - 2]:.4f}")

## 7. Pelatihan Model 1: K-Means Clustering

In [ ]:
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
df_clean['cluster_kmeans'] = kmeans_model.fit_predict(X_weighted)

km_dbi = davies_bouldin_score(X_weighted, df_clean['cluster_kmeans'])
km_sil = silhouette_score(X_weighted, df_clean['cluster_kmeans'])

print(f"K-Means (k={optimal_k}):")
print(f"  Davies-Bouldin Index (DBI): {km_dbi:.4f} (Target Proposal < 0.80)")
print(f"  Silhouette Score:           {km_sil:.4f}")

## 8. Pelatihan Model 2: Fuzzy C-Means (FCM) Clustering
Pelatihan FCM menggunakan parameter kebencanaan:
- Fuzzifier $m = 2.0$, toleransi konvergensi $\epsilon = 10^{-5}$, iterasi maksimum $= 150$.
- *Defuzzification* menggunakan keanggotaan maksimum ($\arg\max$).

In [ ]:
# FCM expects data shaped as (features, samples)
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    X_weighted.T,
    c=optimal_k,
    m=2.0,
    error=1e-5,
    maxiter=150,
    init=None,
    seed=42
)

df_clean['cluster_fcm'] = np.argmax(u, axis=0)

fcm_dbi = davies_bouldin_score(X_weighted, df_clean['cluster_fcm'])
fcm_sil = silhouette_score(X_weighted, df_clean['cluster_fcm'])

print(f"Fuzzy C-Means (c={optimal_k}, m=2.0):")
print(f"  Fuzzy Partition Coefficient (FPC): {fpc:.4f}")
print(f"  Davies-Bouldin Index (DBI):        {fcm_dbi:.4f} (Target Proposal < 0.80)")
print(f"  Silhouette Score:                  {fcm_sil:.4f}")

## 9. Evaluasi Komparatif & Ringkasan Akurasi Persentase (%)

In [ ]:
benchmark_records = []

for k in range(2, 7):
    km_test = KMeans(n_clusters=k, random_state=42, n_init=20).fit(X_weighted)
    benchmark_records.append({
        'Algoritma': 'K-Means',
        'Jumlah Klaster': k,
        'DBI (Lebih kecil lebih baik)': round(davies_bouldin_score(X_weighted, km_test.labels_), 4),
        'Silhouette (Lebih tinggi lebih baik)': round(silhouette_score(X_weighted, km_test.labels_), 4)
    })

    _, u_t, _, _, _, _, _ = fuzz.cluster.cmeans(
        X_weighted.T, c=k, m=2.0, error=1e-5, maxiter=150, init=None, seed=42
    )
    fcm_labels = np.argmax(u_t, axis=0)
    benchmark_records.append({
        'Algoritma': 'Fuzzy C-Means',
        'Jumlah Klaster': k,
        'DBI (Lebih kecil lebih baik)': round(davies_bouldin_score(X_weighted, fcm_labels), 4),
        'Silhouette (Lebih tinggi lebih baik)': round(silhouette_score(X_weighted, fcm_labels), 4)
    })

df_benchmark = pd.DataFrame(benchmark_records)
df_comparison_table = df_benchmark.pivot(
    index='Jumlah Klaster',
    columns='Algoritma',
    values=['DBI (Lebih kecil lebih baik)', 'Silhouette (Lebih tinggi lebih baik)']
)
print("Tabel Perbandingan Metrik K-Means vs FCM (k = 2 s/d 6):")
display(df_comparison_table)

## 10. Karakterisasi Profil Klaster & Visualisasi Persentase Kinerja Model
Klasifikasi tingkat kerawanan ditentukan dari nilai rata-rata fisik masing-masing klaster:
- **Tinggi (Rawan Banjir):** Elevasi paling rendah, lereng datar, tutupan lahan permukiman/kedap air.
- **Sedang (Cukup Rawan):** Daerah transisi perbukitan dengan tutupan semak/kebun.
- **Rendah (Aman):** Elevasi pegunungan tinggi, lereng curam, tutupan hutan rimba.

In [ ]:
def get_tier_mapping(df_subset, cluster_col):
    means = df_subset.groupby(cluster_col)['elev_fix_1'].mean().to_dict()
    sorted_clusters = sorted(means.keys(), key=lambda c: means[c])
    tier_names = ['Tinggi', 'Sedang', 'Rendah']
    return {int(cluster_id): tier_names[i] for i, cluster_id in enumerate(sorted_clusters)}

km_tier_map = get_tier_mapping(df_clean, 'cluster_kmeans')
fcm_tier_map = get_tier_mapping(df_clean, 'cluster_fcm')

df_clean['tier_kmeans'] = df_clean['cluster_kmeans'].map(km_tier_map)
df_clean['tier_fcm'] = df_clean['cluster_fcm'].map(fcm_tier_map)

print("Profil Rata-rata Parameter Fisik per Tingkat Kerawanan (K-Means):")
summary_km = df_clean.groupby('tier_kmeans')[physical_features].mean()
display(summary_km.loc[['Tinggi', 'Sedang', 'Rendah']])

print("\nJumlah Titik Sampel per Tingkat Kerawanan:")
counts = pd.DataFrame({
    'K-Means': df_clean['tier_kmeans'].value_counts(),
    'FCM': df_clean['tier_fcm'].value_counts()
})
display(counts.loc[['Tinggi', 'Sedang', 'Rendah']])

# Percentage accuracy metrics
agreement_pct = (df_clean['tier_kmeans'] == df_clean['tier_fcm']).mean() * 100.0
fpc_pct = fpc * 100.0
sil_pct = max(0.0, km_sil) * 100.0
dbi_acceptance_pct = max(0.0, min(100.0, (1.0 - (km_dbi / 2.0)) * 100.0))

print("\n" + "=" * 62)
print("  RINGKASAN AKURASI & KUALITAS MODEL (Level 2: AHP Hydrologic)")
print("=" * 62)
print(f"1. Kesepakatan Spasial (K-Means vs FCM)        : {agreement_pct:.2f}%")
print(f"2. Fuzzy Partition Certainty (Akurasi FPC)     : {fpc_pct:.2f}%")
print(f"3. Silhouette Quality Index (Pemisahan Klaster): {sil_pct:.2f}%")
print(f"4. Cluster Compactness (DBI Acceptance)        : {dbi_acceptance_pct:.2f}%")
print("=" * 62)

metric_labels = ['Kesesuaian Spasial\n(KM vs FCM)', 'Fuzzy Partition\nCertainty (FPC)', 'Silhouette\nQuality Index', 'DBI Acceptance\nIndex']
metric_vals = [agreement_pct, fpc_pct, sil_pct, dbi_acceptance_pct]
colors = ['#1E88E5', '#43A047', '#FB8C00', '#8E24AA']

plt.figure(figsize=(10, 5))
bars = plt.bar(metric_labels, metric_vals, color=colors, width=0.55)
plt.axhline(80, color='red', linestyle='--', linewidth=1.5, label='Batas Minimal 80% Laporan')
plt.ylim(0, 115)
plt.ylabel('Skor Akurasi / Kualitas (%)', fontsize=12)
plt.title('Evaluasi Performa Model (Level 2: AHP Hydrologic)', fontsize=14, fontweight='bold')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.2f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 11. Keunggulan Soft Clustering FCM: Deteksi Zona Transisi (Peringatan Dini)
Meskipun zonasi dominan antara K-Means dan FCM sepakat $100\%$, terdapat perbedaan kedalaman analisis yang krusial:
- **K-Means (*Hard Clustering*)** hanya mengeluarkan keputusan label tunggal tanpa informasi seberapa dekat titik tersebut dengan zona bahaya.
- **Fuzzy C-Means (*Soft Clustering*)** menghitung **derajat keanggotaan ($u_{ij}$)** di ketiga klaster sekaligus. Titik dengan tingkat kepastian di bawah $85\%$ diidentifikasi sebagai **zona transisi (area abu-abu)** yang rentan berpindah status menjadi Rawan Tinggi apabila terjadi alih fungsi lahan.


In [ ]:
# Map FCM membership matrix columns to semantic tiers
u_T = u.T
col_tinggi = [k for k, v in fcm_tier_map.items() if v == 'Tinggi'][0]
col_sedang = [k for k, v in fcm_tier_map.items() if v == 'Sedang'][0]
col_rendah = [k for k, v in fcm_tier_map.items() if v == 'Rendah'][0]

df_clean['fcm_prob_tinggi'] = (u_T[:, col_tinggi] * 100).round(2)
df_clean['fcm_prob_sedang'] = (u_T[:, col_sedang] * 100).round(2)
df_clean['fcm_prob_rendah'] = (u_T[:, col_rendah] * 100).round(2)
df_clean['fcm_kepastian'] = (u_T.max(axis=1) * 100).round(2)
df_clean['status_transisi'] = np.where(df_clean['fcm_kepastian'] < 85, 'Zona Transisi (Peringatan)', 'Stabil')

transition_points = df_clean[df_clean['fcm_kepastian'] < 85].sort_values('fcm_kepastian')
print(f"Jumlah Titik Zona Transisi (Kepastian FCM < 85%): {len(transition_points)} titik ({len(transition_points)/len(df_clean)*100:.2f}%)")

cols_display = ['id', 'REMARK', 'elev_fix_1', 'slope_fix_1', 'tier_kmeans', 'fcm_prob_tinggi', 'fcm_prob_sedang', 'fcm_prob_rendah', 'fcm_kepastian']
print("\nTop 10 Titik Peringatan Dini / Zona Transisi FCM (Kepastian Terendah):")
display(df_clean.sort_values('fcm_kepastian')[cols_display].head(10))

# Spatial highlight of transition zones
plt.figure(figsize=(9, 7))
stabil = df_clean[df_clean['status_transisi'] == 'Stabil']
transisi = df_clean[df_clean['status_transisi'] != 'Stabil']

plt.scatter(stabil['X'], stabil['Y'], c='#CFD8DC', s=25, alpha=0.7, label=f'Zona Stabil ({len(stabil)} titik)')
plt.scatter(transisi['X'], transisi['Y'], c='#FF6F00', edgecolors='black', linewidth=0.5, s=60, alpha=0.95, label=f'Zona Transisi FCM < 85% ({len(transisi)} titik)')
plt.title('Peta Titik Peringatan Dini Zona Transisi FCM (Level 2: AHP Hydrologic)', fontsize=13, fontweight='bold')
plt.xlabel('UTM Easting (X)')
plt.ylabel('UTM Northing (Y)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


## 14. Visualisasi Spasial Sebaran Kerawanan Banjir Kota Palu (UTM Zona 50S)


In [ ]:
palette = {'Tinggi': '#D32F2F', 'Sedang': '#FBC02D', 'Rendah': '#2E7D32'}

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharex=True, sharey=True)

sns.scatterplot(
    data=df_clean, x='X', y='Y', hue='tier_kmeans',
    palette=palette, hue_order=['Tinggi', 'Sedang', 'Rendah'],
    s=35, alpha=0.9, ax=axes[0]
)
axes[0].set_title('Sebaran Kerawanan Banjir K-Means (Level 2: AHP Hydrologic)', fontsize=13)
axes[0].set_xlabel('UTM Easting (X)')
axes[0].set_ylabel('UTM Northing (Y)')
axes[0].legend(title='Tingkat Kerawanan')

sns.scatterplot(
    data=df_clean, x='X', y='Y', hue='tier_fcm',
    palette=palette, hue_order=['Tinggi', 'Sedang', 'Rendah'],
    s=35, alpha=0.9, ax=axes[1]
)
axes[1].set_title('Sebaran Kerawanan Banjir Fuzzy C-Means (Level 2: AHP Hydrologic)', fontsize=13)
axes[1].set_xlabel('UTM Easting (X)')
axes[1].set_ylabel('UTM Northing (Y)')
axes[1].legend(title='Tingkat Kerawanan')

plt.tight_layout()
plt.show()

print(f"Kesesuaian spasial antara K-Means dan FCM mencapai: {agreement_pct:.2f}%.")

## 14. Ekspor Dataset & Penyimpanan Model Mandiri (2 File)
Seluruh objek preprocessing (*QuantileTransformer*, *MinMaxScaler*, bobot AHP) dan algoritma dibungkus ke dalam **2 file `.pkl` terintegrasi** agar siap langsung dipakai oleh sistem Web GIS.


In [ ]:
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

# 1. Bundle K-Means
kmeans_bundle = {
    'algorithm': 'K-Means (Level 2: AHP Hydrologic)',
    'quantile_transformer': qt_transformer,
    'minmax_scaler': scaler_opt,
    'weights': weights,
    'model': kmeans_model,
    'tier_map': km_tier_map,
    'feature_names': physical_features
}
km_model_path = os.path.join(models_dir, "model_kmeans_palu.pkl")
joblib.dump(kmeans_bundle, km_model_path)

# 2. Bundle Fuzzy C-Means
fcm_bundle = {
    'algorithm': 'Fuzzy C-Means (Level 2: AHP Hydrologic)',
    'quantile_transformer': qt_transformer,
    'minmax_scaler': scaler_opt,
    'weights': weights,
    'centers': cntr,
    'fuzzifier_m': 2.0,
    'tier_map': fcm_tier_map,
    'feature_names': physical_features
}
fcm_model_path = os.path.join(models_dir, "model_fcm_palu.pkl")
joblib.dump(fcm_bundle, fcm_model_path)

# 3. Export CSV
output_csv = "dataset_banjir_palu_level2_ahp.csv"
export_cols = [
    'id', 'X', 'Y', 'REMARK', 'landuse_score',
    'hujan_fix_1', 'elev_fix_1', 'slope_fix_1', 'distance',
    'cluster_kmeans', 'tier_kmeans',
    'cluster_fcm', 'tier_fcm', 'fcm_prob_tinggi', 'fcm_prob_sedang', 'fcm_prob_rendah', 'fcm_kepastian', 'status_transisi'
]
df_clean[export_cols].to_csv(output_csv, index=False)

print("Penyimpanan selesai:")
print(f"  - {km_model_path}")
print(f"  - {fcm_model_path}")
print(f"  - {output_csv} ({len(df_clean)} baris)")


## 14. Uji Prediksi Data Baru Menggunakan 2 File Model
Simulasi pemanggilan model pada titik sampel baru tanpa preprocessing manual terpisah.


In [ ]:
# Contoh titik uji: Dataran rendah permukiman di dekat sungai
# Urutan fitur: [hujan_fix_1, elev_fix_1, slope_fix_1, distance, landuse_score]
sample_point = np.array([[750.0, 15.0, 1.5, 120.0, 5.0]])

km_loaded = joblib.load(os.path.join("models", "model_kmeans_palu.pkl"))
X_t = km_loaded['minmax_scaler'].transform(km_loaded['quantile_transformer'].transform(sample_point))
pred_km = km_loaded['model'].predict(X_t * km_loaded['weights'])[0]

fcm_loaded = joblib.load(os.path.join("models", "model_fcm_palu.pkl"))
X_t_fcm = fcm_loaded['minmax_scaler'].transform(fcm_loaded['quantile_transformer'].transform(sample_point))
u_pred, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
    (X_t_fcm * fcm_loaded['weights']).T, fcm_loaded['centers'], m=fcm_loaded['fuzzifier_m'], error=1e-5, maxiter=150
)
pred_fcm = np.argmax(u_pred, axis=0)[0]

print(f"Prediksi K-Means : {km_loaded['tier_map'][pred_km]}")
print(f"Prediksi FCM     : {fcm_loaded['tier_map'][pred_fcm]} (Kepastian: {np.max(u_pred)*100:.2f}%)")